# Cropy Qwen 기능 선택 모델 직접 해 보기

보드·서버에 올릴 모델(`cropy_qwen_v2/soup_v2_v2b_v2c/merged_hf`)에 발화와 앱 화면 상태를 넣고, 모델이 고른 기능을 바로 본다.

- 런타임: GPU면 아무거나(T4도 됨). 메뉴 런타임 → 모두 실행 → 맨 아래 입력 칸이 나오면 쓴다.
- 화면 상태는 예시에서 고르고, 필요하면 JSON을 직접 고친다(예: 병명, 작물 불일치, 농약 목록).
- **맞음/틀림**을 누르면 `cropy_qwen_v2/demo_log.jsonl`에 남는다. 틀린 문장은 실제 발화 평가 자료로 쓴다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, re, sys, subprocess, time, datetime

DRIVE_DIR = Path('/content/drive/MyDrive/cropy_qwen_v2')
MODEL_DIR = DRIVE_DIR / 'soup_v2_v2b_v2c/merged_hf'
LOG_FILE = DRIVE_DIR / 'demo_log.jsonl'
assert (MODEL_DIR / 'cropy_prompt.json').exists(), f'{MODEL_DIR}가 없습니다. 병합·평균 노트북을 먼저 돌리세요'
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.19.0', 'accelerate==1.15.0',
                       'tokenizers==0.23.2'])
import torch, transformers
from transformers import AutoTokenizer
from collections.abc import Mapping

PROMPT = json.loads((MODEL_DIR / 'cropy_prompt.json').read_text(encoding='utf-8'))
SYSTEM_PROMPT, FUNCTIONS = PROMPT['system_prompt'], PROMPT['functions']
NAMES = list(FUNCTIONS)
MEANINGS = {"get_diagnosis": {"description": "현재 진단 결과를 설명한다. 분류 결과를 확진이나 현장 측정으로 바꾸지 않는다.", "args": {"topic": ["병명 또는 정상 여부", "심각도 표시의 의미", "살펴볼 증상", "결과가 어떻게 나왔는지(모델·출처)", "진행·피해에 미치는 영향", "먹어도 되는지", "이전 결과와 달라진 점"]}}, "get_prediction_metrics": {"description": "모델 점수와 후보를 설명한다. 점수를 실제 발병 확률로 해석하지 않는다.", "args": {"topic": ["점수 수치와 그 숫자가 무슨 점수인지", "후보 순위", "점수 분포·합계", "점수로 알 수 없는 것(발병 확률·피해율·확진으로 해석하는 질문)", "후보끼리 점수 비교"]}}, "get_recapture_guidance": {"description": "현재 진단에 저장된 재촬영 안내를 설명한다.", "args": {"topic": ["결과 화면에서 다시 찍을 부위와 방법", "다시 찍어야 하는 이유", "다시 찍으면 확정되는지", "재촬영 안내가 나온 근거"]}}, "get_care_information": {"description": "제공된 원인·예방·관리 정보를 알려 준다.", "args": {"topic": ["원인", "예방·관리 방법", "무엇부터 관리할지"]}}, "get_pesticide_information": {"description": "진단된 작물·병해에 등록된 농약 정보를 알려 준다. 여러 제품 중 대상이 모호하면 clarify_request(target=product)를 쓴다.", "args": {"topic": ["약 목록·이름", "희석 배수", "사용 방법·시기", "수확 전 사용 금지 기간", "살포 후 재출입 간격", "등록·검증 여부", "성분", "요약", "사용 전 확인할 것", "제품끼리 비교", "다른 약과 섞어 써도 되는지"], "product": ["지금 보고 있는(하나로 확인된) 제품", "목록 전체"]}}, "clarify_request": {"description": "대상이 모호하거나 서로 충돌하면 추측하지 않고 확인 질문을 한다.", "args": {"target": ["어떤 작물인지", "어떤 병인지", "무엇을 원하는지(서로 다른 요청을 한꺼번에 한 경우 포함)", "어느 점수인지", "심각도가 무엇을 뜻하는지", "어느 기록인지", "어느 제품인지", "잘못 알아들은 말"]}}, "handle_scope_limit": {"description": "진단 기능과 관련 있지만 지원하지 않는 요청에 지원 범위를 안내한다.", "args": {"reason": ["고추·무·배추가 아닌 작물", "앱이 실행할 수 없는 요청(물 주기·주문·기기 조작 등)", "작물 사진이 아닌 입력"]}}, "get_usage_guidance": {"description": "앱 사용 방법과 다음에 할 일을 안내한다.", "args": {"topic": ["지금 다음에 할 일", "처음 사진 찍는 방법", "작물 고르는 방법", "분석이 어떻게 되고 있는지", "다시 시도하는 방법", "방금 안내를 다시"], "delivery": ["짧게 한 번에", "한 단계씩"]}}, "chit_chat": {"description": "인사·감사·날씨·시간·일상 대화처럼 작물 진단·농약·앱 사용과 무관한 말에 짧게 응대한다.", "args": {}}}
PRESETS = {"P01_no_photo": {"ui_stage": "no_photo"}, "P02_photo_ready": {"ui_stage": "photo_ready", "selected_crop": "배추"}, "P03_analyzing": {"ui_stage": "analyzing", "selected_crop": "무"}, "P04_error": {"ui_stage": "error"}, "P05_result_disease_one_product": {"ui_stage": "result", "selected_crop": "고추", "predicted_crop": "고추", "crop_mismatch": false, "prediction_mode": "model", "category": "disease", "disease_name": "고추 탄저병", "confidence": 0.91, "top_predictions": [{"name": "고추 탄저병", "confidence": 0.91}, {"name": "고추 흰가루병", "confidence": 0.05}], "severity": "high", "pesticides": ["포룸디 (디메토모르프.디티아논 수화제)"]}, "P06_result_disease_many_products": {"ui_stage": "result", "selected_crop": "배추", "predicted_crop": "배추", "crop_mismatch": false, "prediction_mode": "model", "category": "disease", "disease_name": "배추 노균병", "confidence": 0.84, "pesticides": ["메가폰 (가스가마이신 입상수화제)", "차트라 (가스가마이신.폴리옥신디 입상수화제)", "미리스타 (디메토모르프 수화제)"]}, "P07_result_confusion_recapture": {"ui_stage": "result", "selected_crop": "무", "predicted_crop": "무", "crop_mismatch": false, "prediction_mode": "model", "category": "disease", "disease_name": "무 노균병", "confidence": 0.51, "top_predictions": [{"name": "무 노균병", "confidence": 0.51}, {"name": "무 검은무늬병", "confidence": 0.46}], "decision_status": "confusion", "recapture": {"region": "잎 뒷면"}}, "P08_result_normal": {"ui_stage": "result", "selected_crop": "배추", "predicted_crop": "배추", "crop_mismatch": false, "prediction_mode": "model", "category": "normal", "disease_name": "배추 정상", "confidence": 0.97}, "P09_result_crop_mismatch": {"ui_stage": "result", "selected_crop": "고추", "predicted_crop": "무", "crop_mismatch": true, "prediction_mode": "model", "category": "disease", "disease_name": "무 검은무늬병", "confidence": 0.72, "decision_status": "crop_mismatch"}, "P10_result_low_confidence": {"ui_stage": "result", "selected_crop": "무", "predicted_crop": "무", "crop_mismatch": false, "prediction_mode": "model", "category": "disease", "disease_name": "무 검은무늬병", "confidence": 0.38, "decision_status": "low_confidence"}, "P11_result_save_failed": {"ui_stage": "result", "selected_crop": "고추", "predicted_crop": "고추", "crop_mismatch": false, "prediction_mode": "model", "category": "disease", "disease_name": "고추 흰가루병", "confidence": 0.88, "save_status": "failed"}}
assert list(MEANINGS) == NAMES
CALL = re.compile(r'^<hs_(\d+)>\(((?:[a-z_]+=-?\d+(?:, [a-z_]+=-?\d+)*)?)\)<hs_end>$')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE == 'cuda' else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
END_ID = tokenizer.convert_tokens_to_ids(PROMPT['stop_token'])
model = getattr(transformers, 'Qwen3_5ForConditionalGeneration').from_pretrained(
    MODEL_DIR, device_map={'': 0} if DEVICE == 'cuda' else None, dtype=DTYPE, attn_implementation='sdpa').eval()
print('불러옴', MODEL_DIR, DEVICE, DTYPE)

In [ ]:
def explain(text):
    m = CALL.match(text.strip())
    if not m or int(m.group(1)) >= len(NAMES):
        return '형식 오류: 서버가 해석할 수 없는 답'
    name = NAMES[int(m.group(1))]
    given = [p.split('=') for p in m.group(2).split(', ')] if m.group(2) else []
    if [k for k, _ in given] != [a for a, _ in FUNCTIONS[name]]:
        return f'형식 오류: {name}의 인자 이름이 다름'
    lines = [f'{name} — {MEANINGS[name]["description"]}']
    for k, v in given:
        values = MEANINGS[name]['args'][k]
        if not 1 <= int(v) <= len(values):
            return f'형식 오류: {name}.{k}={v}는 없는 번호'
        lines.append(f'  {k}={v}: {values[int(v) - 1]}')
    return '\n'.join(lines)

def ask(utterance, context):
    user = json.dumps({'utterance': utterance, 'context': context}, ensure_ascii=False, separators=(',', ':'))
    ids = tokenizer.apply_chat_template([{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user}],
                                        tokenize=True, add_generation_prompt=True, enable_thinking=False)
    if isinstance(ids, Mapping):
        ids = ids['input_ids']
    ids = ids.tolist() if hasattr(ids, 'tolist') else list(ids)
    ids = ids[0] if ids and isinstance(ids[0], list) else ids
    start = time.time()
    with torch.no_grad():
        out = model.generate(input_ids=torch.tensor([ids], device=model.device), max_new_tokens=16, do_sample=False,
                             eos_token_id=[END_ID, tokenizer.eos_token_id], pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    seq = out[0, len(ids):].tolist()
    seq = seq[:seq.index(END_ID) + 1] if END_ID in seq else seq
    return tokenizer.decode(seq, skip_special_tokens=False).strip(), len(ids), time.time() - start

print(ask('안녕하세요', PRESETS['P01_no_photo']))

## 입력 칸
발화를 쓰고 **실행**. 답이 맞는지 보고 **맞음/틀림**을 누르면 기록된다(틀림이면 메모에 기대한 답을 적어 두면 좋다).

In [ ]:
import ipywidgets as w
from IPython.display import display

utter = w.Text(placeholder='예: 이거 무슨 병이야', description='발화', layout=w.Layout(width='95%'))
preset = w.Dropdown(options=list(PRESETS), value='P05_result_disease_one_product', description='화면 상태')
ctx = w.Textarea(value=json.dumps(PRESETS[preset.value], ensure_ascii=False, indent=1), description='context',
                 layout=w.Layout(width='95%', height='170px'))
note = w.Text(placeholder='틀렸을 때 기대한 답이나 메모', description='메모', layout=w.Layout(width='95%'))
run, good, bad = w.Button(description='실행', button_style='primary'), w.Button(description='맞음'), w.Button(description='틀림')
out = w.Output()
last = {}

def on_preset(change):
    ctx.value = json.dumps(PRESETS[change['new']], ensure_ascii=False, indent=1)

def on_run(_):
    out.clear_output()
    with out:
        try:
            context = json.loads(ctx.value)
        except json.JSONDecodeError as e:
            print('context JSON 오류:', e); return
        if not utter.value.strip():
            print('발화를 입력하세요'); return
        answer, n, sec = ask(utter.value.strip(), context)
        last.clear(); last.update(utterance=utter.value.strip(), preset=preset.value, context=context, output=answer)
        print(answer)
        print(explain(answer))
        print(f'(입력 {n}토큰, {sec:.2f}초, {model.device})')

def mark(ok):
    def handler(_):
        with out:
            if not last:
                print('먼저 실행하세요'); return
            row = dict(last, correct=ok, note=note.value.strip(), at=datetime.datetime.now().isoformat(timespec='seconds'))
            with LOG_FILE.open('a', encoding='utf-8') as f:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
            print('기록함:', '맞음' if ok else '틀림', '→', LOG_FILE.name)
            last.clear(); note.value = ''
    return handler

preset.observe(on_preset, names='value')
run.on_click(on_run)
if hasattr(utter, 'on_submit'):  # Enter로 실행 (ipywidgets 7)
    utter.on_submit(on_run)
good.on_click(mark(True)); bad.on_click(mark(False))
display(w.VBox([utter, preset, ctx, w.HBox([run, good, bad]), note, out]))

## 기록 보기

In [ ]:
if LOG_FILE.exists():
    rows = [json.loads(l) for l in LOG_FILE.read_text(encoding='utf-8').splitlines()]
    print(f'{len(rows)}건, 맞음 {sum(r["correct"] for r in rows)}건')
    for r in rows[-20:]:
        print('O' if r['correct'] else 'X', r['preset'], '|', r['utterance'], '→', r['output'], '|', r['note'])
else:
    print('아직 기록이 없습니다')